# GLD_BUILD_AGGREGATES
**Layer:** Gold  
**Purpose:** Build pre-aggregated Gold summary tables consumed by reporting and dashboards:
- `gld_agg_txn_hourly`  — hourly transaction KPIs with anomaly counts and P95 response time
- `gld_agg_auth_daily`  — daily per-customer authentication summary

**Pattern:** Overwrite matching partitions for idempotency (no MERGE required for aggregates).

## 1. Parameters

In [ ]:
batch_id            = "dev-run-00000000"
storage_account     = "adlsbankingdev"
gold_container      = "gold"
keyvault_name       = "kv-banking-dev"
# Date range to (re-)aggregate — overwrite matching partitions
watermark_date      = "2024-01-01"   # inclusive lower bound (YYYY-MM-DD)
run_date            = "2024-01-02"   # exclusive upper bound (YYYY-MM-DD)

## 2. Imports and Spark Configuration

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, LongType, DoubleType, StringType
import datetime

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "64")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
spark.conf.set("spark.sql.adaptive.enabled", "true")
# Allow overwrite of individual partitions without touching others
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

def adls_path(container, *parts):
    base = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
    return "/".join([base] + list(parts))

gold_delta_base     = adls_path(gold_container, "delta")
ingestion_timestamp = datetime.datetime.utcnow().isoformat() + "Z"

# Convert date strings to integer date surrogate keys for filter pushdown
wm_sk  = int(watermark_date.replace("-", ""))
run_sk = int(run_date.replace("-", ""))

print(f"batch_id={batch_id}  window=[{watermark_date}, {run_date})")
print(f"date_sk window: [{wm_sk}, {run_sk})")
print(f"Gold base: {gold_delta_base}")

## 3. Read Source Gold Fact Tables

In [ ]:
# ── gld_fact_transaction_logs ─────────────────────────────────────────────────
fact_txn = (
    spark.read.format("delta")
    .load(f"{gold_delta_base}/gld_fact_transaction_logs")
    .filter(
        (F.col("event_date_sk") >= F.lit(wm_sk)) &
        (F.col("event_date_sk") <  F.lit(run_sk))
    )
)

# ── gld_fact_auth_events ──────────────────────────────────────────────────────
fact_auth = (
    spark.read.format("delta")
    .load(f"{gold_delta_base}/gld_fact_auth_events")
    .filter(
        (F.col("event_date_sk") >= F.lit(wm_sk)) &
        (F.col("event_date_sk") <  F.lit(run_sk))
    )
)

print(f"fact_txn  rows: {fact_txn.cache().count():,}")
print(f"fact_auth rows: {fact_auth.cache().count():,}")

## 4. Build gld_agg_txn_hourly

In [ ]:
# Derive hour_of_day from event_timestamp
fact_txn_h = fact_txn.withColumn(
    "hour_of_day", F.hour(F.col("event_timestamp")).cast(IntegerType())
)

# Approximate P95 using percentile_approx (no external dependency)
agg_txn_hourly = (
    fact_txn_h
    .groupBy(
        "event_date_sk",
        "hour_of_day",
        "channel_sk",
        "txn_type_sk",
        "service_sk"
    )
    .agg(
        F.count("*")                                          .alias("total_transactions"),
        F.sum(
            F.when(F.col("status_code") == "SUCCESS", 1).otherwise(0)
        )                                                     .alias("successful_transactions"),
        F.sum(
            F.when(F.col("status_code") != "SUCCESS", 1).otherwise(0)
        )                                                     .alias("failed_transactions"),
        F.sum(F.col("amount").cast(DoubleType()))             .alias("total_amount"),
        F.avg(F.col("response_time_ms").cast(DoubleType()))   .alias("avg_response_time_ms"),
        F.sum(
            F.when(F.col("is_anomaly") == True, 1).otherwise(0)
        )                                                     .alias("anomaly_count"),
        F.percentile_approx(
            F.col("response_time_ms").cast(DoubleType()), 0.95
        )                                                     .alias("p95_response_time_ms"),
    )
    .withColumn("agg_batch_id",             F.lit(batch_id))
    .withColumn("agg_ingestion_timestamp",  F.lit(ingestion_timestamp))
)

row_count = agg_txn_hourly.count()
print(f"gld_agg_txn_hourly: {row_count:,} rows")

## 5. Write gld_agg_txn_hourly — Overwrite Partitions

In [ ]:
agg_txn_hourly_path = f"{gold_delta_base}/gld_agg_txn_hourly"

(
    agg_txn_hourly.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere",
        f"event_date_sk >= {wm_sk} AND event_date_sk < {run_sk}")
    .partitionBy("event_date_sk")
    .save(agg_txn_hourly_path)
)
print(f"gld_agg_txn_hourly written to {agg_txn_hourly_path}")

## 6. Build gld_agg_auth_daily

In [ ]:
agg_auth_daily = (
    fact_auth
    .groupBy(
        "event_date_sk",
        "customer_sk",
        "channel_sk"
    )
    .agg(
        F.count("*")                                              .alias("total_login_attempts"),
        F.sum(
            F.when(F.col("auth_result") == "SUCCESS", 1).otherwise(0)
        )                                                         .alias("successful_logins"),
        F.sum(
            F.when(F.col("auth_result") == "FAILURE", 1).otherwise(0)
        )                                                         .alias("failed_logins"),
        F.sum(
            F.when(F.col("auth_result") == "LOCKED",  1).otherwise(0)
        )                                                         .alias("locked_events"),
        # Approximate distinct IPs (countDistinct is exact but can be expensive)
        F.countDistinct(F.col("ip_address"))                      .alias("unique_ip_count"),
        F.countDistinct(F.col("device_id"))                       .alias("unique_device_count"),
        F.max(F.col("is_high_velocity").cast(IntegerType()))      .alias("has_high_velocity_flag"),
    )
    # Derive failure rate
    .withColumn("failure_rate",
        F.when(F.col("total_login_attempts") > 0,
            F.col("failed_logins").cast(DoubleType()) /
            F.col("total_login_attempts").cast(DoubleType())
        ).otherwise(F.lit(0.0)))
    .withColumn("agg_batch_id",             F.lit(batch_id))
    .withColumn("agg_ingestion_timestamp",  F.lit(ingestion_timestamp))
)

auth_row_count = agg_auth_daily.count()
print(f"gld_agg_auth_daily: {auth_row_count:,} rows")

## 7. Write gld_agg_auth_daily — Overwrite Partitions

In [ ]:
agg_auth_daily_path = f"{gold_delta_base}/gld_agg_auth_daily"

(
    agg_auth_daily.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere",
        f"event_date_sk >= {wm_sk} AND event_date_sk < {run_sk}")
    .partitionBy("event_date_sk")
    .save(agg_auth_daily_path)
)
print(f"gld_agg_auth_daily written to {agg_auth_daily_path}")

## 8. Summary

In [ ]:
print("=" * 60)
print("GLD_BUILD_AGGREGATES complete")
print(f"  batch_id           : {batch_id}")
print(f"  window             : [{watermark_date}, {run_date})")
print(f"  txn_hourly rows    : {row_count:,}")
print(f"  auth_daily rows    : {auth_row_count:,}")
print(f"  finished           : {ingestion_timestamp}")
print("=" * 60)